In [1]:
# This cell will load all my Walmart data from the local CSV file.
import pandas as pd
df = pd.read_csv("Walmart_Sales.csv")
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [2]:
pip install pymongo

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Makes column names lowercase and easier to work with
df.columns = df.columns.str.lower()
df.columns

Index(['store', 'date', 'weekly_sales', 'holiday_flag', 'temperature',
       'fuel_price', 'cpi', 'unemployment'],
      dtype='object')

In [4]:
# Just checking the shape and data types of the dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   store         6435 non-null   int64  
 1   date          6435 non-null   object 
 2   weekly_sales  6435 non-null   float64
 3   holiday_flag  6435 non-null   int64  
 4   temperature   6435 non-null   float64
 5   fuel_price    6435 non-null   float64
 6   cpi           6435 non-null   float64
 7   unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


In [5]:
# My first source: CSV - extract tables
fact_sales = df[['store', 'date', 'weekly_sales']]

dim_date = df[['date']].drop_duplicates().reset_index(drop=True)
dim_date['year'] = pd.to_datetime(dim_date['date']).dt.year
dim_date['month'] = pd.to_datetime(dim_date['date']).dt.month
dim_date['day'] = pd.to_datetime(dim_date['date']).dt.day
dim_date['quarter'] = pd.to_datetime(dim_date['date']).dt.quarter

dim_weather = df[['temperature', 'holiday_flag']].drop_duplicates()
dim_economic = df[['fuel_price', 'cpi', 'unemployment']].drop_duplicates()

In [6]:
# My second source: MySQL - store dimension data
import mysql.connector

mysql_conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='your_mysql_password',
    database='walmart_stores'
)

dim_store = pd.read_sql("SELECT * FROM dim_store_info", mysql_conn)
mysql_conn.close()
dim_store.head()

/var/folders/c0/18tbtyvs0tlcv80bwlq50q080000gn/T/ipykernel_18799/792451799.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_store = pd.read_sql("SELECT * FROM dim_store_info", mysql_conn)


,store,store_type,store_size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [7]:
# My third source: MongoDB Atlas
from pymongo import MongoClient

mongo_client =  MongoClient("your_mongodb_connection_string")
db = mongo_client["walmart_db"]
collection = db["dim_economic"]

collection.drop()
collection.insert_many(dim_economic.to_dict(orient='records'))

mongo_df = pd.DataFrame(list(collection.find({}, {'_id': 0})))
mongo_df.head()

,fuel_price,cpi,unemployment
0,2.572,211.096358,8.106
1,2.548,211.242170,8.106
2,2.514,211.289143,8.106
3,2.561,211.319643,8.106
4,2.625,211.350143,8.106


In [8]:
# Saves tables to sqlite as the final data mart destination
import sqlite3

conn = sqlite3.connect('sales.db')

fact_sales.to_sql('fact_sales', conn, if_exists='replace', index=False)
dim_store.to_sql('dim_store', conn, if_exists='replace', index=False)
dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)
dim_weather.to_sql('dim_weather', conn, if_exists='replace', index=False)
dim_economic.to_sql('dim_economic', conn, if_exists='replace', index=False)
mongo_df.to_sql('dim_economic_mongo', conn, if_exists='replace', index=False)

4574

In [9]:
# Query 1: Total sales per store
query = """
SELECT f.store, SUM(f.weekly_sales) as total_sales
FROM fact_sales f
JOIN dim_store s ON f.store = s.store
JOIN dim_date d ON f.date = d.date
GROUP BY f.store
"""

pd.read_sql(query, conn)

,store,total_sales
0,1,2.224028e+08
1,2,2.753824e+08
2,3,5.758674e+07
3,4,2.995440e+08
4,5,4.547569e+07
5,6,2.237561e+08
6,7,8.159828e+07
7,8,1.299512e+08
8,9,7.778922e+07
9,10,2.716177e+08


In [10]:
# Query 2: Average weekly sales by store type
query2 = """
SELECT s.store_type, AVG(f.weekly_sales) as avg_sales
FROM fact_sales f
JOIN dim_store s ON f.store = s.store
GROUP BY s.store_type
"""
pd.read_sql(query2, conn)

,store_type,avg_sales
0,A,1.376673e+06
1,B,8.229950e+05
2,C,4.726148e+05


In [11]:
# Query 3: Measuring the average temperature during holiday weeks vs non-holiday weeks
query3 = """
SELECT holiday_flag, COUNT(*) as num_weeks, AVG(temperature) as avg_temp
FROM dim_weather
GROUP BY holiday_flag
"""
pd.read_sql(query3, conn)

,holiday_flag,num_weeks,avg_temp
0,0,3329,59.589751
1,1,342,49.157427
